# Is my strategy's timing real?

[`position_test`](../functions_fin/position_test.md) keeps your strategy's
trades and moves them to random times, then reruns each copy through the same
backtest engine. The p-value is the share of random timings that do at least as
well as yours. A small p-value says the timing carries information.

The data: one-minute bars of six hours of BTC perpetual trades.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from screamer import (Resample, EwMean, LogReturn, Sign, FillNa, BacktestOHLCTarget,
                      position_test)

trades = pd.read_csv("data/deribit_btc_perp_6h.csv")
bars, _ = Resample(freq=60_000, agg="ohlc")(trades["price"].to_numpy(),
                                             trades["timestamp"].to_numpy())
o, h, l, c = bars.T

## Test a strategy

In [ ]:
signal = Sign()(FillNa(0.0)(EwMean(span=10)(LogReturn()(c))))   # follow the recent trend
engine = BacktestOHLCTarget(taker_fee=5e-4)

res = position_test(engine, signal, o, h, l, c, n=999)
print(f"Sharpe per bar {res.observed:+.3f}, p = {res.p_value:.3f}")

## Look at every statistic

The p-value covers one statistic, the Sharpe ratio. `res.null_stats` holds every
[`backtest_report`](../functions_fin/backtest_report.md) statistic of each
random timing, and `res.observed_stats` those of the real run, so you can place
the real run on two statistics at once. Each dot is one random timing; the star
is your strategy.

In [ ]:
null, real = res.null_stats, res.observed_stats
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, (x, y) in zip(axes, [("sharpe", "max_drawdown"), ("total_pnl", "win_rate")]):
    ax.scatter(null[x], null[y], s=6, color="lightsteelblue", label="random timing")
    ax.scatter(real[x], real[y], s=250, marker="*", color="crimson", label="your strategy")
    ax.set_xlabel(x); ax.set_ylabel(y)
axes[0].legend(fontsize=8)
plt.tight_layout()

The star sits inside the cloud on both views: six hours of data give no
evidence that this trend rule's timing beats chance. The
[significance recipe](../recipes/significance-test.md) runs the same test on a
market where the rule has an edge.

## Shift or segments

`method="shift"` (the default) rotates the whole position series by a random
offset, so the trades keep their order and clustering. `method="segments"`
shuffles the order of the trades, which also breaks their clustering. Use it to
check that the result does not hinge on a few trades in one favourable
stretch.

In [ ]:
for method in ("shift", "segments"):
    p = position_test(engine, signal, o, h, l, c, n=999, method=method).p_value
    print(f"{method:8s} p = {p:.3f}")

## Adapt it

- Pass your own target series and engine: `position_test(engine, target, *market)`
  takes the same arguments as `engine(target, *market)`.
- Choose the statistic with `metric=` (`"total_pnl"`, `"sortino"`,
  `"max_drawdown"`, ...) and annualize ratios with `periods_per_year=`.
- Tried several variants? Pass them as one `(T, K)` target and read
  `family_p_value`.
- Tuned the strategy on this data? `position_test` cannot see the tuning; see
  [I tuned my parameters. Is the result real?](21-is-the-tuned-result-real.ipynb).